# Lesson 7 — Model comparison: information criteria, cross-validation and Bayesian model selection

Fitting tells you how well *one* model can account for the data. Science usually needs the next question: which of several candidate explanations should we believe? A model with more parameters can always fit at least as well, so raw fit cannot decide. This lesson covers the three families of tools used in the field — **information criteria** (AIC, BIC), **cross-validation**, and **Bayesian model selection** at the level of a group of participants — and ends where lesson 6 left off: a model that wins the comparison can still be wrong, and **posterior predictive checks** are how you find out.

**What you will learn**

- why more parameters always fit better, and why that is a problem (overfitting),
- what AIC and BIC are, where they come from, and how to read their differences,
- how to cross-validate a *sequential* learning model without destroying its learning history,
- how to compare models across a group: fixed effects versus random effects, the variational algorithm of Stephan et al. (2009), exceedance probabilities and their protected version (Rigoux et al., 2014),
- how to combine relative comparison with absolute checks.

**Prerequisites:** lessons 5 and 6.

**Estimated time:** 2.5 hours.

In [ ]:
import sys; sys.path.insert(0, "..")      # find the rlcourse package when running from notebooks/
import numpy as np
import matplotlib.pyplot as plt
from rlcourse import check, hint, reveal, progress, use, setup_style
from rlcourse import models, tasks, fitting, plotting
setup_style()
rng = np.random.default_rng(2024)        # one seed per notebook: results are reproducible

## 7.1 The problem: more parameters always fit better

Take the RW model of lesson 3 (parameters $\alpha$, $\beta$) and the asymmetric RW model of Lefebvre et al. (2017), which has separate learning rates for positive and negative prediction errors ($\alpha^+$, $\alpha^-$, $\beta$). The second model **nests** the first: with $\alpha^+ = \alpha^-$ it *is* the first. Whatever the data, the best-fitting asymmetric model therefore has a negative log-likelihood at most equal to the best-fitting RW model — even when the data were generated by plain RW and the asymmetry is pure noise.

In [ ]:
n_subjects, n_trials = 20, 150
sims, nll_rw, nll_asym = [], np.zeros(n_subjects), np.zeros(n_subjects)
for i in range(n_subjects):
    alpha, beta = rng.uniform(0.1, 0.9), np.exp(rng.uniform(np.log(1), np.log(8)))
    sim = models.simulate_model("rw", (alpha, beta), [0.7, 0.3], n_trials, rng)        # generated by plain RW
    sims.append(sim)
    data = (sim["choices"], sim["rewards"])
    nll_rw[i] = fitting.fit_mle(models.nll_rw, data, ["unit", "pos"], n_starts=2, rng=rng)["nll"]
    nll_asym[i] = fitting.fit_mle(models.nll_rw_asym, data, ["unit", "unit", "pos"], n_starts=2, rng=rng)["nll"]

improvement = nll_rw - nll_asym
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.bar(np.arange(n_subjects), improvement)
ax.set(xlabel="simulated subject", ylabel="nll(RW) − nll(asymmetric RW)",
       title="The bigger model never fits worse, even when the data come from the smaller one")
ax.grid(axis="x", visible=False)
plt.show()
print(f"asymmetric RW has the lower nll for {np.mean(improvement >= -1e-6):.0%} of subjects; "
      f"mean improvement {improvement.mean():.2f} nats")

The improvement is small but never negative. If we picked models by fit alone we would conclude, for every single subject, that learning is asymmetric — when by construction it is not. This is **overfitting**: the extra parameter is absorbing random fluctuations in the choices, and a model that fits noise predicts new data *worse*.

Every method in this lesson is a way of estimating how much of the fit is signal and how much is noise.

## 7.2 Information criteria

### AIC

Akaike (1974) asked how well a fitted model would predict *new* data from the same process. The in-sample log-likelihood is an optimistic estimate of that, and the bias turns out to be approximately the number of fitted parameters $k$. Correcting for it, and multiplying by $-2$ for historical reasons, gives the **Akaike information criterion**:

$$
\mathrm{AIC} = 2k + 2\,\mathrm{nll} .
$$

### BIC

Schwarz (1978) started from a Bayesian question: how probable are the data under each model, averaged over all its parameter values — the **marginal likelihood** or **model evidence** $p(y \mid M) = \int p(y \mid \theta, M)\,p(\theta \mid M)\,d\theta$? For large $n$ this integral is approximated by the fit at the maximum and a penalty that grows with the sample size:

$$
\mathrm{BIC} = k \ln n + 2\,\mathrm{nll}, \qquad \ln p(y \mid M) \approx -\tfrac{1}{2}\,\mathrm{BIC}.
$$

Here $n$ is the number of data points that enter the likelihood — the number of choices (for the two-step task of lesson 9, both stages count). Lower is better for both criteria, and **only differences between models fitted to the same data are meaningful**. For BIC the difference has a direct interpretation: $\Delta\mathrm{BIC}/2$ approximates the log **Bayes factor**, and Kass and Raftery (1995) suggest reading $\Delta\mathrm{BIC}$ as

| ΔBIC | evidence for the better model |
|---|---|
| 0 – 2 | barely worth mentioning |
| 2 – 6 | positive |
| 6 – 10 | strong |
| > 10 | very strong |

The two criteria differ in their penalty: 2 per parameter for AIC, $\ln n$ per parameter for BIC. Since $\ln n > 2$ as soon as $n > 7$, BIC is the more conservative criterion for any realistic data set, and the two can disagree. They also answer different questions — AIC "which model predicts best?", BIC "which model most probably generated the data?" — so when they disagree, report both.

### ✏️ Exercise 7.1 — AIC and BIC *(coding)*

Implement both criteria. The function receives the negative log-likelihood `nll`, the number of free parameters `k` and the number of data points `n`, and returns the tuple `(AIC, BIC)`.

In [ ]:
def information_criteria(nll, k, n):
    """Return (AIC, BIC) for a fitted model with negative log-likelihood nll, k parameters and n data points."""
    aic = ...                # <- 2k + 2 nll
    bic = ...                # <- k ln(n) + 2 nll
    return aic, bic

check("7.1", information_criteria)

In [ ]:
information_criteria = use("7.1", information_criteria)

aic_rw, bic_rw = information_criteria(nll_rw, 2, n_trials)
aic_asym, bic_asym = information_criteria(nll_asym, 3, n_trials)
print(f"AIC prefers plain RW (the true model) for {np.mean(aic_rw < aic_asym):.0%} of subjects")
print(f"BIC prefers plain RW (the true model) for {np.mean(bic_rw < bic_asym):.0%} of subjects")

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.bar(np.arange(n_subjects) - 0.2, aic_asym - aic_rw, 0.4, label="ΔAIC")
ax.bar(np.arange(n_subjects) + 0.2, bic_asym - bic_rw, 0.4, label="ΔBIC")
ax.axhline(0, color="#52514e", lw=0.8)
ax.set(xlabel="simulated subject", ylabel="asymmetric RW − RW  (positive favours RW)",
       title="With a complexity penalty the true model wins")
ax.legend()
ax.grid(axis="x", visible=False)
plt.show()

### ✏️ Exercise 7.2 — BIC by hand *(maths)*

A model with 3 free parameters was fitted to 200 choices and reached a negative log-likelihood of 100. Compute its BIC (two decimals).

In [ ]:
answer = None   # <- your number
check("7.2", answer)

### ✏️ Exercise 7.3 — From a BIC difference to a Bayes factor *(maths)*

Model A has a BIC 6 points lower than model B. Using $\ln p(y \mid M) \approx -\mathrm{BIC}/2$, what is the approximate Bayes factor in favour of A, that is, the ratio $p(y \mid A)/p(y \mid B)$? Give one decimal.

In [ ]:
answer = None   # <- your number
check("7.3", answer)

### ✏️ Exercise 7.4 — AIC versus BIC *(conceptual)*

For a data set of $n = 150$ choices, why does BIC penalise an extra parameter more heavily than AIC?

**(a)** Because BIC multiplies the negative log-likelihood by $n$.
**(b)** Because BIC's penalty per parameter is $\ln n \approx 5.0$, larger than AIC's 2; this is the case whenever $n > e^2 \approx 7.4$.
**(c)** Because BIC uses base-10 logarithms.
**(d)** It does not: AIC always penalises more.

In [ ]:
check("7.4", None)   # <- replace None with "a", "b", "c" or "d"

## 7.3 Cross-validation

Information criteria *estimate* out-of-sample performance from the in-sample fit. **Cross-validation** measures it: fit the model on part of the data (the training set) and evaluate the likelihood of the rest (the test set). Overfitting is penalised automatically, because parameters that fit noise in the training set do not help with the test set.

For learning models there is a catch. A trial's predicted choice probability depends on everything the participant experienced before it, so you cannot cut out the test trials and evaluate them in isolation: the model would start every test block with blank values. The correct procedure is to **replay the whole sequence** — updating the values on every trial, training and test alike — but to **score only the test trials**, that is, to sum $-\log p(c_t)$ only over $t$ in the test set. Blocks of consecutive trials (or whole sessions) are used as folds rather than randomly scattered trials, so that the test trials are not sandwiched between training trials of the same learning episode.

### ✏️ Exercise 7.5 — Scoring held-out trials *(coding)*

Modify the RW likelihood of lesson 5 (`models.nll_rw`) so that it replays the whole sequence but sums the negative log-likelihood only over the trials where `test_mask[t]` is `True`.

In [ ]:
def nll_rw_on_trials(params, choices, rewards, test_mask, n_arms=2, q0=0.5):
    """Negative log-likelihood of the RW + softmax model on the trials where test_mask is True,
    replaying the whole sequence so that the learning history is intact.  params = (alpha, beta)."""
    alpha, beta = params
    Q = np.full(n_arms, q0, dtype=float)
    nll = 0.0
    for t, (c, r) in enumerate(zip(choices, rewards)):
        raise NotImplementedError("your code here: compute p, score the trial only if it is a test trial, then update Q[c]")
    return nll

check("7.5", nll_rw_on_trials)

Let us cross-validate the nested comparison of section 7.1: train on the first half of each subject's trials, test on the second half. For the training fit we simply pass the first half of the data to `fit_mle`. For the test score we need a model-specific "score only these trials" function like the one you just wrote; for the asymmetric model we use a generic (slower) trick instead — the per-trial contribution of trial $t$ is the difference between the cumulative negative log-likelihood up to and including $t$ and the one up to $t-1$.

In [ ]:
nll_rw_on_trials = use("7.5", nll_rw_on_trials)

def per_trial_nll(nll_fn, params, choices, rewards):
    """-log p(c_t) for every trial, by differencing cumulative nll values (generic, O(T^2), fine for small T)."""
    cumulative = np.array([nll_fn(params, choices[:t], rewards[:t]) for t in range(len(choices) + 1)])
    return np.diff(cumulative)

n_train = n_trials // 2
test_mask = np.arange(n_trials) >= n_train
cv_rw, cv_asym, train_fits_rw = np.zeros(n_subjects), np.zeros(n_subjects), []
for i, sim in enumerate(sims):
    c, r = sim["choices"], sim["rewards"]
    fit_rw = fitting.fit_mle(models.nll_rw, (c[:n_train], r[:n_train]), ["unit", "pos"], n_starts=2, rng=rng)
    fit_asym = fitting.fit_mle(models.nll_rw_asym, (c[:n_train], r[:n_train]), ["unit", "unit", "pos"], n_starts=2, rng=rng)
    cv_rw[i] = nll_rw_on_trials(fit_rw["params"], c, r, test_mask)
    cv_asym[i] = per_trial_nll(models.nll_rw_asym, fit_asym["params"], c, r)[test_mask].sum()
    train_fits_rw.append(fit_rw["params"])

# the generic trick and the specific function agree (subject 0, RW model):
generic = per_trial_nll(models.nll_rw, train_fits_rw[0], sims[0]["choices"], sims[0]["rewards"])[test_mask].sum()
print(f"RW test-set nll of subject 0: specific function {cv_rw[0]:.4f}, generic differencing {generic:.4f}")

diff = cv_asym - cv_rw
print(f"\ntest-set nll (second half): RW mean {cv_rw.mean():.1f}, asymmetric RW mean {cv_asym.mean():.1f}")
print(f"asymmetric − RW: median {np.median(diff):+.2f} nats, mean {diff.mean():+.2f} nats;  "
      f"RW predicts better for {np.mean(diff > 0):.0%} of subjects;  "
      f"the asymmetric model is worse by more than 5 nats for {np.sum(diff > 5)} subject(s)")

Out of sample the picture is very different from section 7.1, where the bigger model fitted better for *every* subject. For the typical subject the two models now predict the held-out trials about equally well — the median difference is close to zero — and on average the smaller model wins, because for a few subjects the asymmetric model's training fit chased noise so hard (a learning rate at 0 or 1 combined with an enormous β) that it predicts the second half very poorly. Overfitting shows up as *worse prediction*, which is exactly what cross-validation measures. Two remarks:

- Cross-validation makes no distributional assumptions and works for any model, but it is noisier than information criteria when the test set is small (the per-subject differences above have a heavy tail), and it costs a fit per fold. Several folds (e.g. leave-one-block-out) reduce the noise.
- In fully Bayesian analyses the same idea appears as leave-one-out cross-validation estimated from posterior samples (Vehtari, Gelman & Gabry, 2017), which we meet again in lesson 8.

## 7.4 Comparing models across a group

So far every comparison was per subject. With a group of participants we want a group-level conclusion, and there are two ways to get one.

**Fixed effects.** Assume that every participant uses the same model, so that the group evidence is the product of the individual evidences: $\ln p(y \mid M_k) = \sum_n L_{nk}$, where $L_{nk} = \ln p(y_n \mid M_k) \approx -\mathrm{BIC}_{nk}/2$. This is just the summed BIC. It is simple, but a single participant with overwhelming evidence can decide the outcome for everyone, and the assumption that everybody uses the same model is often wrong — strategies differ between people.

**Random effects** (Stephan, Penny, Daunizeau, Moran & Friston, 2009). Treat the model each participant uses as a random variable drawn from a population with unknown frequencies $r = (r_1, \dots, r_K)$:

$$
r \sim \mathrm{Dirichlet}(\alpha_0), \qquad m_n \sim \mathrm{Categorical}(r), \qquad y_n \sim p(y_n \mid m_n).
$$

The quantities of interest are the posterior over the frequencies $r$ (how common is each model?) and, for each participant, the posterior over which model they use. Stephan et al. derived a variational approximation in which the posterior over $r$ is again a Dirichlet, $\mathrm{Dirichlet}(\alpha)$, and the posterior over participant $n$'s model is a vector $g_n$. The two are updated alternately until they stop changing:

$$
\text{E-step:}\quad u_{nk} = \exp\!\Big(L_{nk} + \psi(\alpha_k) - \psi\big(\textstyle\sum_j \alpha_j\big)\Big), \qquad g_{nk} = \frac{u_{nk}}{\sum_j u_{nj}}
$$

$$
\text{M-step:}\quad \alpha_k = \alpha_{0,k} + \sum_n g_{nk}
$$

where $\psi$ is the digamma function, and $\psi(\alpha_k) - \psi(\sum_j \alpha_j)$ is the expected log-frequency $\mathbb{E}[\ln r_k]$ under the current Dirichlet. In words: given the current belief about how common each model is, assign each participant to the models in proportion to their evidence; given the assignments, update the counts. The prior $\alpha_0 = 1$ for every model says that, before seeing data, all frequencies are equally plausible.

From $\alpha$ we read off the **expected frequency** of each model, $\langle r_k \rangle = \alpha_k / \sum_j \alpha_j$, and the **exceedance probability**

$$
\varphi_k = P\big(r_k > r_j \ \forall j \neq k \mid y\big),
$$

the probability that model $k$ is the most common one in the population. It is easy to estimate by sampling $r$ from $\mathrm{Dirichlet}(\alpha)$ many times and counting how often each model has the largest frequency.

### ✏️ Exercise 7.6 — Random-effects Bayesian model selection *(coding)*

Implement the variational iteration. `log_evidence` is an array of shape `(n_subjects, n_models)` with $L_{nk}$; return the Dirichlet parameters `alpha`. `scipy.special.digamma` is $\psi$ and `scipy.special.logsumexp` normalises safely in log space.

In [ ]:
from scipy.special import digamma, logsumexp

def bms_alpha(log_evidence, alpha0=1.0, max_iter=500, tol=1e-8):
    """Variational Bayesian model selection (Stephan et al. 2009): returns the Dirichlet parameters alpha."""
    L = np.asarray(log_evidence, dtype=float)
    n_subjects, n_models = L.shape
    alpha = np.full(n_models, float(alpha0))
    for _ in range(max_iter):
        log_u = ...                                              # <- E-step: L + ψ(alpha) − ψ(sum alpha)
        g = ...                                                  # <- normalise within each subject (axis=1), in log space
        alpha_new = ...                                          # <- M-step: alpha0 + expected counts
        if np.max(np.abs(alpha_new - alpha)) < tol:
            alpha = alpha_new
            break
        alpha = alpha_new
    return alpha

check("7.6", bms_alpha)

### ✏️ Exercise 7.7 — Exceedance probabilities *(coding)*

Estimate the exceedance probabilities by Monte Carlo: draw `n_samples` frequency vectors from $\mathrm{Dirichlet}(\alpha)$ with `rng.dirichlet(alpha, size=n_samples)`, find the most frequent model in each draw, and return the proportion of draws in which each model is the most frequent. (Because this is a Monte Carlo estimate, the checker allows a tolerance of a few percent.)

In [ ]:
def exceedance_probabilities(alpha, n_samples, rng):
    """P(r_k > r_j for all j != k) under Dirichlet(alpha), estimated from n_samples draws."""
    alpha = np.asarray(alpha, dtype=float)
    samples = ...                                   # <- draws of r, shape (n_samples, n_models)
    winners = ...                                   # <- index of the largest frequency in each draw
    return ...                                      # <- proportion of draws won by each model

check("7.7", exceedance_probabilities)

### Protected exceedance probabilities

Exceedance probabilities have a known weakness (Rigoux, Stephan, Friston & Daunizeau, 2014): they describe which model is *more* frequent, but not whether the models differ in frequency at all. With enough participants, even a tiny, chance difference in evidence gives one model a high $\varphi_k$. Rigoux et al. therefore compute the **Bayes omnibus risk** (BOR), the posterior probability of the null hypothesis that all models are equally frequent, by comparing the evidence of the full random-effects model with that of the null model, and define the **protected exceedance probability**

$$
\tilde\varphi_k = (1 - \mathrm{BOR})\,\varphi_k + \mathrm{BOR}/K ,
$$

which falls back to chance ($1/K$) when the data do not support any difference. `fitting.bms` computes $\alpha$, the expected frequencies, $\varphi$, BOR and $\tilde\varphi$ in one call; the cell below uses it alongside your own functions.

### A group study

Twenty simulated participants, of whom twelve learn by plain RW, five follow win-stay/lose-shift and three learn asymmetrically. Four candidate models are fitted to everyone; the log evidence is approximated by $-\mathrm{BIC}/2$.

In [ ]:
bms_alpha = use("7.6", bms_alpha)
exceedance_probabilities = use("7.7", exceedance_probabilities)

def sample_params(name, rng):
    """Draw plausible generating parameters for each candidate model."""
    beta = np.exp(rng.uniform(np.log(1), np.log(8)))
    if name == "bias":
        return (rng.uniform(0.2, 0.8),)
    if name == "wsls":
        return (rng.uniform(0.05, 0.4),)
    if name == "rw":
        return (rng.uniform(0.1, 0.9), beta)
    if name == "rw_asym":
        return (rng.uniform(0.1, 0.9), rng.uniform(0.1, 0.9), beta)

generating = ["rw"] * 12 + ["wsls"] * 5 + ["rw_asym"] * 3
candidates = ["bias", "wsls", "rw", "rw_asym"]
labels = {"bias": "bias", "wsls": "WSLS", "rw": "RW", "rw_asym": "RW-asym"}
n_trials_group = 150
group_nll = np.zeros((len(generating), len(candidates)))
for i, gen_name in enumerate(generating):
    data = models.simulate_model(gen_name, sample_params(gen_name, rng), [0.7, 0.3], n_trials_group, rng)
    for j, cand in enumerate(candidates):
        nll_fn, param_names, kinds = models.MODEL_SPECS[cand]
        group_nll[i, j] = fitting.fit_mle(nll_fn, (data["choices"], data["rewards"]), kinds, n_starts=2, rng=rng)["nll"]

k = np.array([len(models.MODEL_SPECS[c][1]) for c in candidates])
group_bic = k * np.log(n_trials_group) + 2 * group_nll
log_evidence = -group_bic / 2

alpha = bms_alpha(log_evidence)
expected_r = alpha / alpha.sum()
xp = exceedance_probabilities(alpha, 20000, rng)
res = fitting.bms(log_evidence, rng=rng)                  # also returns the protected version

print("fixed effects (summed BIC, lower is better):", {labels[c]: round(float(v), 1) for c, v in zip(candidates, group_bic.sum(axis=0))})
print("random effects:")
print(f"   {'model':8s} {'alpha':>7s} {'E[r]':>7s} {'XP':>7s} {'PXP':>7s}")
for j, cand in enumerate(candidates):
    print(f"   {labels[cand]:8s} {alpha[j]:7.2f} {expected_r[j]:7.2f} {xp[j]:7.2f} {res['pxp'][j]:7.2f}")
print(f"   Bayes omnibus risk = {res['bor']:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), width_ratios=[1.6, 1])
im = axes[0].imshow(res["g"].T, aspect="auto", cmap=plotting.SEQUENTIAL, vmin=0, vmax=1)
axes[0].set(yticks=range(len(candidates)), yticklabels=[labels[c] for c in candidates], xlabel="participant",
            title="Posterior probability of each model per participant (g)")
axes[0].set_xticks(range(len(generating)))
axes[0].set_xticklabels([labels[g][:3] for g in generating], rotation=90, fontsize=7)
axes[0].grid(False)
plt.colorbar(im, ax=axes[0], fraction=0.03)
x = np.arange(len(candidates))
axes[1].bar(x - 0.2, expected_r, 0.4, label="expected frequency E[r]")
axes[1].bar(x + 0.2, res["pxp"], 0.4, label="protected exceedance prob.")
axes[1].set(xticks=x, xticklabels=[labels[c] for c in candidates], ylim=(0, 1.05), title="Group-level results")
axes[1].legend(fontsize=8)
axes[1].grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

The x-axis labels of the left panel show which model actually generated each participant's data (the first three letters), so you can see that the per-participant posteriors mostly land on the right model. At the group level the expected frequencies recover the mixture (roughly 60% RW, 25% WSLS, 15% asymmetric), and the exceedance probability says that RW is almost certainly the most common model — a correct conclusion that a fixed-effects analysis would have expressed only as "RW wins", hiding the fact that a quarter of the group does something else.

### When the exceedance probability misleads

Now a group in which **no model is better**: the log evidences are pure noise. The unprotected exceedance probabilities still single out a "winner"; the protected ones do not.

In [ ]:
noise_evidence = rng.normal(0, 0.5, size=(40, 3))              # 40 subjects, 3 models, no real differences
res_null = fitting.bms(noise_evidence, rng=rng)
print("expected frequencies:        ", np.round(res_null["r"], 2))
print("exceedance probabilities:    ", np.round(res_null["xp"], 2))
print("Bayes omnibus risk:          ", round(res_null["bor"], 3))
print("protected exceedance probs.: ", np.round(res_null["pxp"], 2))

### ✏️ Exercise 7.8 — Expected model frequency *(maths)*

A random-effects analysis of three models returned $\alpha = (12.3,\ 4.2,\ 1.5)$. What is the expected frequency of model 1 in the population? Give three decimals.

In [ ]:
answer = None   # <- your number
check("7.8", answer)

### ✏️ Exercise 7.9 — What an exceedance probability means *(conceptual)*

An exceedance probability of 0.90 for model B means ...

**(a)** 90% of the participants are best described by model B.
**(b)** The posterior probability that model B is more frequent in the population than any other candidate model is 0.90.
**(c)** Model B is the true model with probability 0.90.
**(d)** Model B's parameters are recovered with 90% accuracy.

In [ ]:
check("7.9", None)   # <- replace None with "a", "b", "c" or "d"

## 7.5 Winning is not being right: posterior predictive checks

Every method above is **relative**: it ranks the candidates you supplied. If the true process is not among them, the winner is merely the least wrong, and a protected exceedance probability of 0.99 says nothing about whether it captures how people learn. Palminteri, Wyart and Koechlin (2017) argue that model comparison must be paired with **falsification**: simulate data from the winning model with its fitted parameters and check whether it reproduces the qualitative patterns in the data — a **posterior predictive check** (lesson 6).

Here is the full workflow on the participant of lesson 6, who follows a noisy win-stay/lose-shift rule. Suppose we did not think of that rule and compared only a bias model, RW and asymmetric RW.

In [ ]:
def stay_after(choices, rewards):
    """P(repeat the previous choice | previous trial rewarded) and (| previous trial unrewarded)."""
    stay = choices[1:] == choices[:-1]
    prev_reward = rewards[:-1]
    return stay[prev_reward == 1].mean(), stay[prev_reward == 0].mean()

def compare_candidates(names, data, rng, n_starts=3):
    """Fit every candidate; return ({name: fit}, {name: BIC})."""
    fits, bics = {}, {}
    for name in names:
        nll_fn, param_names, kinds = models.MODEL_SPECS[name]
        fits[name] = fitting.fit_mle(nll_fn, (data["choices"], data["rewards"]), kinds, n_starts=n_starts, rng=rng)
        bics[name] = fitting.bic(fits[name]["nll"], len(param_names), len(data["choices"]))
    return fits, bics

def posterior_predictive(name, params, data, rng, n_sims=200):
    """Stay statistics of the data and of n_sims data sets simulated from the fitted model."""
    data_stats = stay_after(data["choices"], data["rewards"])
    sim_stats = np.array([stay_after(**models.simulate_model(name, params, [0.7, 0.3], len(data["choices"]), rng))
                          for _ in range(n_sims)])
    return np.array(data_stats), sim_stats

labels["rw_stick"] = "RW+stick"
participant = models.simulate_model("wsls", (0.1,), [0.7, 0.3], 200, rng)          # the participant of lesson 6

first_set = ["bias", "rw", "rw_asym"]
fits1, bics1 = compare_candidates(first_set, participant, rng)
winner1 = min(bics1, key=bics1.get)
print("candidate set 1:", {labels[c]: round(float(v), 1) for c, v in bics1.items()},
      f"-> winner {labels[winner1]} (ΔBIC to runner-up {sorted(bics1.values())[1] - bics1[winner1]:.1f})")
data_stats, sim_stats1 = posterior_predictive(winner1, fits1[winner1]["params"], participant, rng)

RW wins decisively. Now the check: does the winner reproduce how the participant behaves after wins and after losses?

In [ ]:
def plot_ppc(ax, data_stats, sim_stats, label):
    x = np.arange(2)
    ax.bar(x - 0.18, data_stats, 0.36, color="#52514e", label="data")
    ax.bar(x + 0.18, sim_stats.mean(axis=0), 0.36, label=label,
           yerr=np.abs(np.percentile(sim_stats, [2.5, 97.5], axis=0) - sim_stats.mean(axis=0)), capsize=4)
    ax.set(xticks=x, xticklabels=["after a win", "after a loss"], ylabel="P(repeat choice)", ylim=(0, 1.05))
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(axis="x", visible=False)

fig, ax = plt.subplots(figsize=(5.5, 3.8))
plot_ppc(ax, data_stats, sim_stats1, f"winner of set 1 ({labels[winner1]}), simulated")
ax.set_title("Posterior predictive check of the winning model")
plt.show()

The winner cannot produce the participant's systematic switching after a loss. The comparison was decisive and the model is still wrong — and the discrepancy points at the missing ingredient: a mechanism by which a loss drives a switch. That is how new candidates enter the set. Two models have such a mechanism: RW with (negative) choice stickiness, and win-stay/lose-shift itself. Add them and compare again.

In [ ]:
second_set = ["bias", "rw", "rw_asym", "rw_stick", "wsls"]
fits2, bics2 = compare_candidates(second_set, participant, rng)
winner2 = min(bics2, key=bics2.get)
print("candidate set 2:", {labels[c]: round(float(v), 1) for c, v in bics2.items()},
      f"-> winner {labels[winner2]} (ΔBIC to runner-up {sorted(bics2.values())[1] - bics2[winner2]:.1f})")
_, sim_stats2 = posterior_predictive(winner2, fits2[winner2]["params"], participant, rng)

fig, ax = plt.subplots(figsize=(5.5, 3.8))
plot_ppc(ax, data_stats, sim_stats2, f"winner of set 2 ({labels[winner2]}), simulated")
ax.set_title("The new winner passes the check")
plt.show()

The expanded comparison picks the right mechanism by a wide margin and the new winner reproduces the data. Note what the two steps did: model comparison chose *among* the candidates, the posterior predictive check told us whether the candidates were *adequate*. Only when the winning model reproduces the patterns you care about is the comparison worth reporting.

### ✏️ Exercise 7.10 — Winning is not being right *(conceptual, written)*

Model B has the lowest BIC for every participant and a protected exceedance probability of 0.99. Explain, in 2–4 sentences, why this does not establish that model B describes how participants actually learn, and what you would do about it.

In [ ]:
check("7.10", """
    Write your answer here.
""")

## 7.6 Putting it together

A complete model-comparison report in a paper from this field typically contains, for every candidate model:

- the number of parameters and the mean (or summed) BIC across participants — the fixed-effects view;
- the expected frequency and the protected exceedance probability from a random-effects analysis;
- evidence that the comparison *could* have identified the right model: the confusion/inversion matrix of lesson 6;
- a posterior predictive check of the winning model on the behavioural signatures that motivated the study.

The cell below assembles the first two items for the group study above.

In [ ]:
import pandas as pd

report = pd.DataFrame({
    "model": [labels[c] for c in candidates],
    "k": k,
    "mean BIC": group_bic.mean(axis=0).round(1),
    "summed BIC": group_bic.sum(axis=0).round(1),
    "E[r]": res["r"].round(2),
    "XP": res["xp"].round(2),
    "PXP": res["pxp"].round(2),
})
report

## Summary

- A model with more parameters never fits worse; raw likelihood cannot be used to compare models of different complexity.
- **AIC** ($2k + 2\,\mathrm{nll}$) estimates predictive accuracy; **BIC** ($k \ln n + 2\,\mathrm{nll}$) approximates the log model evidence. Only differences matter; $\Delta\mathrm{BIC}/2 \approx \ln$ Bayes factor.
- **Cross-validation** measures out-of-sample fit directly. For learning models, replay the whole sequence and score only the held-out trials.
- **Random-effects Bayesian model selection** treats the model as varying across participants and yields expected frequencies and **exceedance probabilities**; the **protected** version guards against declaring a winner when no model is better.
- Relative comparison must be paired with **absolute** checks: simulate from the winning model and verify that it reproduces the qualitative patterns in the data.

## What's next

Lesson 8 goes fully Bayesian: priors, maximum a posteriori estimation, Markov chain Monte Carlo and hierarchical models in which the group informs every individual estimate — the machinery behind the "model evidence" we approximated here with the BIC.

## References

- Akaike, H. (1974). A new look at the statistical model identification. *IEEE Transactions on Automatic Control, 19*(6), 716–723. https://doi.org/10.1109/TAC.1974.1100705
- Schwarz, G. (1978). Estimating the dimension of a model. *Annals of Statistics, 6*(2), 461–464. https://doi.org/10.1214/aos/1176344136
- Kass, R. E., & Raftery, A. E. (1995). Bayes factors. *Journal of the American Statistical Association, 90*(430), 773–795. https://doi.org/10.1080/01621459.1995.10476572
- Stephan, K. E., Penny, W. D., Daunizeau, J., Moran, R. J., & Friston, K. J. (2009). Bayesian model selection for group studies. *NeuroImage, 46*(4), 1004–1017. https://doi.org/10.1016/j.neuroimage.2009.03.025
- Rigoux, L., Stephan, K. E., Friston, K. J., & Daunizeau, J. (2014). Bayesian model selection for group studies — revisited. *NeuroImage, 84*, 971–985. https://doi.org/10.1016/j.neuroimage.2013.08.065
- Vehtari, A., Gelman, A., & Gabry, J. (2017). Practical Bayesian model evaluation using leave-one-out cross-validation and WAIC. *Statistics and Computing, 27*, 1413–1432. https://doi.org/10.1007/s11222-016-9696-4
- Palminteri, S., Wyart, V., & Koechlin, E. (2017). The importance of falsification in computational cognitive modeling. *Trends in Cognitive Sciences, 21*(6), 425–433. https://doi.org/10.1016/j.tics.2017.03.011
- Wilson, R. C., & Collins, A. G. E. (2019). Ten simple rules for the computational modeling of behavioral data. *eLife, 8*, e49547. https://doi.org/10.7554/eLife.49547
- Piray, P., Dezfouli, A., Heskes, T., Frank, M. J., & Daw, N. D. (2019). Hierarchical Bayesian inference for concurrent model fitting and comparison for group studies. *PLoS Computational Biology, 15*(6), e1007043. https://doi.org/10.1371/journal.pcbi.1007043
- Lefebvre, G., Lebreton, M., Meyniel, F., Bourgeois-Gironde, S., & Palminteri, S. (2017). Behavioural and neural characterization of optimistic reinforcement learning. *Nature Human Behaviour, 1*, 0067. https://doi.org/10.1038/s41562-017-0067